# Lab 3: Production Deployment of AI Agents

## Why this matters
The course deploys agents to HuggingFace Spaces (great for demos). But production means:
- **Reliability** — agents that don't crash under load
- **Scalability** — handling multiple concurrent users
- **Configurability** — no hardcoded secrets, environment-aware config
- **Restartability** — agents that resume state after a crash
- **Cost control** — budget limits, fallback models, caching

## What we'll cover
1. **FastAPI wrapping** — expose your agent as a REST API
2. **Async & concurrent request handling**
3. **Rate limiting & cost controls**
4. **Streaming responses** — real-time agent output
5. **Docker containerization** — the pattern (no Docker required to follow along)
6. **Health checks & graceful shutdown**

---

In [ ]:
# Install dependencies
# pip install fastapi uvicorn httpx asyncio

import os
import asyncio
import time
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()
aclient = AsyncOpenAI()
print("Async OpenAI client ready.")

---
## Part 1: Wrapping an Agent as a FastAPI Service

Your agent needs to be callable via HTTP — this is how other services, frontends, and orchestrators interact with it.

In [ ]:
# agent_service.py — save this and run with: uvicorn agent_service:app --reload

AGENT_SERVICE_CODE = '''
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import Optional, AsyncGenerator
from openai import AsyncOpenAI
import time, asyncio, os
from dotenv import load_dotenv

load_dotenv()
app = FastAPI(title="Agent Service", version="1.0.0")
client = AsyncOpenAI()

# --- Request/Response models ---
class ChatRequest(BaseModel):
    message: str
    session_id: Optional[str] = None
    stream: bool = False

class ChatResponse(BaseModel):
    answer: str
    session_id: Optional[str]
    latency_ms: float
    model: str

# --- Thread-safe in-memory session store with TTL (use Redis in prod) ---
# asyncio.Lock ensures concurrent requests for the same session_id don\'t race.
# Sessions expire after SESSION_TTL_SECONDS of inactivity — prevents unbounded growth.
sessions: dict[str, list] = {}
session_last_used: dict[str, float] = {}
_session_lock = asyncio.Lock()

SESSION_TTL_SECONDS = int(os.getenv("SESSION_TTL_SECONDS", "3600"))  # 1 hour default

async def _prune_expired_sessions():
    """Remove sessions idle for longer than SESSION_TTL_SECONDS."""
    now = time.time()
    async with _session_lock:
        expired = [sid for sid, last in session_last_used.items()
                   if now - last > SESSION_TTL_SECONDS]
        for sid in expired:
            sessions.pop(sid, None)
            session_last_used.pop(sid, None)
    if expired:
        print(f"[session] Pruned {len(expired)} expired sessions")

# --- Health check ---
@app.get("/health")
async def health():
    await _prune_expired_sessions()
    return {"status": "ok", "timestamp": time.time(), "active_sessions": len(sessions)}

# --- Non-streaming chat endpoint ---
@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    start = time.time()
    
    # Safely read and write session history under a lock
    async with _session_lock:
        history = list(sessions.get(req.session_id, [])) if req.session_id else []
        history.append({"role": "user", "content": req.message})
    
    messages = [
        {"role": "system", "content": "You are a helpful AI engineering assistant."},
        *history
    ]
    
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    
    answer = response.choices[0].message.content
    history.append({"role": "assistant", "content": answer})
    
    # Persist session (trim to last 20 messages to control context size)
    if req.session_id:
        async with _session_lock:
            sessions[req.session_id] = history[-20:]
            session_last_used[req.session_id] = time.time()
    
    return ChatResponse(
        answer=answer,
        session_id=req.session_id,
        latency_ms=(time.time() - start) * 1000,
        model="gpt-4o-mini"
    )

# --- Streaming chat endpoint ---
async def stream_agent(message: str) -> AsyncGenerator[str, None]:
    stream = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": message}],
        stream=True
    )
    async for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            yield f"data: {delta}\\\\n\\\\n"  # Server-Sent Events format
    yield "data: [DONE]\\\\n\\\\n"

@app.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    return StreamingResponse(
        stream_agent(req.message),
        media_type="text/event-stream"
    )

# --- Graceful shutdown ---
@app.on_event("shutdown")
async def shutdown_event():
    print("Saving session state before shutdown...")
    # In prod: flush sessions to Redis/DB here
'''

with open("agent_service.py", "w") as f:
    f.write(AGENT_SERVICE_CODE)

# Verify the file was written and show key sections
import os
size = os.path.getsize("agent_service.py")
print(f"agent_service.py written ({size} bytes).")
print()
print("To run the service:")
print("  uvicorn agent_service:app --reload --port 8000")
print()
print("To test the /health endpoint:")
print("  curl http://localhost:8000/health")
print()
print("To test /chat:")
print("  curl -X POST http://localhost:8000/chat \\")
print('         -H "Content-Type: application/json" \\')
print('         -d \'{"message": "What is LangGraph?", "session_id": "test-1"}\'')
print()
print("Key design points:")
print("  1. asyncio.Lock — concurrent requests don't corrupt the same session")
print("  2. session_last_used tracks idle time for TTL-based pruning")
print("  3. _prune_expired_sessions() is called on each /health poll")
print("  4. SESSION_TTL_SECONDS is configurable via env var (default 1 hour)")

---
## Part 2: Rate Limiting & Cost Controls

In production, you **must** protect against:
- Single users burning your entire API budget
- Runaway agent loops calling tools thousands of times
- Surprise bills from the LLM provider

In [ ]:
import time
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict

# --- Token budget tracker ---
@dataclass
class CostTracker:
    """Track token usage and enforce budgets per session."""
    # OpenAI pricing (USD per 1M tokens) — update as prices change
    INPUT_COST_PER_1M  = {"gpt-4o": 2.50,  "gpt-4o-mini": 0.15}
    OUTPUT_COST_PER_1M = {"gpt-4o": 10.00, "gpt-4o-mini": 0.60}
    
    session_costs: Dict[str, float] = field(default_factory=lambda: defaultdict(float))
    session_calls: Dict[str, int]   = field(default_factory=lambda: defaultdict(int))
    
    # Budget limits
    MAX_COST_PER_SESSION = 0.50   # $0.50 per session
    MAX_CALLS_PER_SESSION = 100   # 100 API calls max
    
    def record(self, session_id: str, model: str, input_tokens: int, output_tokens: int):
        cost = (input_tokens  / 1_000_000 * self.INPUT_COST_PER_1M.get(model, 0.15) +
                output_tokens / 1_000_000 * self.OUTPUT_COST_PER_1M.get(model, 0.60))
        self.session_costs[session_id] += cost
        self.session_calls[session_id] += 1
        return cost
    
    def check_budget(self, session_id: str) -> tuple[bool, str]:
        cost = self.session_costs[session_id]
        calls = self.session_calls[session_id]
        
        if cost >= self.MAX_COST_PER_SESSION:
            return False, f"Session budget exceeded: ${cost:.4f} >= ${self.MAX_COST_PER_SESSION}"
        if calls >= self.MAX_CALLS_PER_SESSION:
            return False, f"Call limit exceeded: {calls} >= {self.MAX_CALLS_PER_SESSION}"
        return True, "ok"
    
    def report(self, session_id: str):
        return {
            "session_id": session_id,
            "total_cost_usd": round(self.session_costs[session_id], 6),
            "api_calls": self.session_calls[session_id]
        }

# Demo
tracker = CostTracker()
tracker.record("user-123", "gpt-4o-mini", input_tokens=500, output_tokens=200)
tracker.record("user-123", "gpt-4o-mini", input_tokens=300, output_tokens=150)

ok, reason = tracker.check_budget("user-123")
print(f"Budget OK: {ok} ({reason})")
print(f"Usage: {tracker.report('user-123')}")

In [ ]:
# --- Async-safe token-bucket rate limiter ---
# The lock makes this safe to use inside async FastAPI handlers.
# A sync rate limiter without a lock can return inconsistent counts under concurrency.
@dataclass
class RateLimiter:
    """Token bucket rate limiter: N requests per window seconds.
    Uses asyncio.Lock so it's safe inside async FastAPI handlers.
    """
    max_requests: int = 10     # 10 requests...
    window_seconds: float = 60  # ...per minute
    
    _buckets: Dict[str, list] = field(default_factory=lambda: defaultdict(list))
    _lock: asyncio.Lock = field(default_factory=asyncio.Lock)
    
    async def is_allowed_async(self, client_id: str) -> tuple[bool, int]:
        """Async-safe check — use this inside FastAPI route handlers."""
        async with self._lock:
            return self._check(client_id)
    
    def is_allowed(self, client_id: str) -> tuple[bool, int]:
        """Sync check — safe for non-async contexts only (e.g. notebook demos)."""
        return self._check(client_id)
    
    def _check(self, client_id: str) -> tuple[bool, int]:
        now = time.time()
        window_start = now - self.window_seconds
        
        self._buckets[client_id] = [
            t for t in self._buckets[client_id] if t > window_start
        ]
        
        if len(self._buckets[client_id]) >= self.max_requests:
            retry_after = int(self._buckets[client_id][0] + self.window_seconds - now) + 1
            return False, retry_after
        
        self._buckets[client_id].append(now)
        remaining = self.max_requests - len(self._buckets[client_id])
        return True, remaining

limiter = RateLimiter(max_requests=3, window_seconds=10)  # 3 per 10s for demo

for i in range(5):
    allowed, info = limiter.is_allowed("user-abc")
    if allowed:
        print(f"Request {i+1}: ALLOWED ({info} remaining)")
    else:
        print(f"Request {i+1}: RATE LIMITED (retry in {info}s)")

print("\nIn a FastAPI handler, use: allowed, info = await limiter.is_allowed_async(client_id)")

---
## Part 3: Streaming Responses

Users hate staring at a blank screen for 10 seconds. Streaming shows tokens as they arrive — critical for agent UX.

In [ ]:
import asyncio
from openai import AsyncOpenAI

aclient = AsyncOpenAI()

async def stream_response(prompt: str):
    """Stream tokens to the console as they arrive."""
    print("Streaming: ", end="", flush=True)
    
    stream = await aclient.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )
    
    full_response = ""
    async for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            print(delta, end="", flush=True)
            full_response += delta
    
    print()  # newline at end
    return full_response

# Run the streaming demo
result = await stream_response("Explain LangGraph in 2 sentences.")
print(f"\nTotal chars received: {len(result)}")

In [ ]:
# --- Concurrent agent calls (how to handle multiple users at once) ---

async def agent_call(user_id: str, question: str) -> tuple[str, float]:
    start = time.time()
    response = await aclient.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}]
    )
    latency = time.time() - start
    return user_id, latency

async def handle_concurrent_users():
    users = [
        ("user-1", "What is CrewAI?"),
        ("user-2", "What is LangGraph?"),
        ("user-3", "What is AutoGen?"),
        ("user-4", "What is MCP?"),
    ]
    
    start_total = time.time()
    # return_exceptions=True means one failure won't cancel the others —
    # each result is either a value or an Exception instance you can inspect
    results = await asyncio.gather(
        *[agent_call(uid, q) for uid, q in users],
        return_exceptions=True
    )
    total = time.time() - start_total
    
    print("Concurrent results:")
    for uid, result in zip([u for u, _ in users], results):
        if isinstance(result, Exception):
            print(f"  {uid}: ERROR — {result}")
        else:
            _, latency = result
            print(f"  {uid}: {latency:.2f}s")
    print(f"Total wall time: {total:.2f}s")

await handle_concurrent_users()

---
## Part 4: Model Fallback Strategy

Production agents need fallbacks when:
- The primary model is overloaded (429 rate limit)
- The primary model is down (503 service unavailable)
- You want to reduce costs by routing simple queries to cheaper models

In [ ]:
from openai import RateLimitError, APIStatusError
import asyncio

MODEL_FALLBACK_CHAIN = [
    "gpt-4o",        # primary: most capable
    "gpt-4o-mini",   # fallback 1: cheaper, fast
    "gpt-3.5-turbo", # fallback 2: last resort
]

async def resilient_llm_call(messages: list, max_retries: int = 3) -> tuple[str, str]:
    """Try models in fallback chain. Returns (answer, model_used)."""
    last_error = None
    
    for model in MODEL_FALLBACK_CHAIN:
        for attempt in range(max_retries):
            try:
                response = await aclient.chat.completions.create(
                    model=model,
                    messages=messages,
                    timeout=30.0
                )
                return response.choices[0].message.content, model
            
            except RateLimitError:
                wait = 2 ** attempt  # exponential backoff: 1s, 2s, 4s
                print(f"Rate limited on {model}, waiting {wait}s...")
                await asyncio.sleep(wait)
                last_error = f"RateLimitError on {model}"
            
            except APIStatusError as e:
                print(f"API error on {model}: {e.status_code}")
                last_error = f"APIError {e.status_code} on {model}"
                break  # try next model immediately
            
            except Exception as e:
                print(f"Unexpected error on {model}: {e}")
                last_error = str(e)
                break
    
    raise RuntimeError(f"All models failed. Last error: {last_error}")

# Test the fallback mechanism
answer, model_used = await resilient_llm_call([
    {"role": "user", "content": "What is 2 + 2?"}
])
print(f"Answer: {answer}")
print(f"Model used: {model_used}")

---
## Part 5: Docker Pattern for Agent Services

You don't need Docker running to understand this — it's the **containerization pattern** that matters.

In [ ]:
DOCKERFILE = '''# Production Dockerfile for an agent service

# Stage 1: Build dependencies
FROM python:3.12-slim AS builder
WORKDIR /app

# Install uv for fast dependency management
RUN pip install uv
COPY pyproject.toml uv.lock ./
RUN uv sync --frozen --no-dev

# Stage 2: Production image (no build tools)
FROM python:3.12-slim AS production
WORKDIR /app

# Non-root user for security
RUN useradd --create-home agent
USER agent

# Copy only what\'s needed
COPY --from=builder /app/.venv /app/.venv
COPY agent_service.py .

# Environment (never hardcode secrets in Dockerfile)
ENV OPENAI_API_KEY=""
ENV PORT=8000
ENV SESSION_TTL_SECONDS=3600

# Worker count: formula is 2 * CPU_CORES + 1.
# Set WEB_CONCURRENCY here as the default; override per deployment env.
# Examples: 2-core dev server → WEB_CONCURRENCY=5
#           4-core prod server → WEB_CONCURRENCY=9
#           8-core prod server → WEB_CONCURRENCY=17
# Or let the platform set it: Railway/Fly.io inject $WEB_CONCURRENCY automatically.
ENV WEB_CONCURRENCY=4

# Health check (Docker will restart if this fails)
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
    CMD curl -f http://localhost:$PORT/health || exit 1

EXPOSE $PORT
CMD ["/app/.venv/bin/uvicorn", "agent_service:app", \\
     "--host", "0.0.0.0", "--port", "8000", \\
     "--workers", "${WEB_CONCURRENCY}"]
'''

# docker-compose: Redis is now actually wired into the agent service via REDIS_URL
DOCKER_COMPOSE = '''# docker-compose.yml for local dev
version: "3.9"

services:
  agent:
    build: .
    ports:
      - "8000:8000"
    environment:
      - OPENAI_API_KEY=${OPENAI_API_KEY}
      - REDIS_URL=redis://redis:6379/0   # wired to the redis service below
      - WEB_CONCURRENCY=4
      - SESSION_TTL_SECONDS=3600
    volumes:
      - ./data:/app/data
    depends_on:
      - redis
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      retries: 3

  redis:
    image: redis:7-alpine
    ports:
      - "6379:6379"
    command: redis-server --appendonly yes
    volumes:
      - redis_data:/data

volumes:
  redis_data:
'''

with open("Dockerfile", "w") as f:
    f.write(DOCKERFILE)
with open("docker-compose.yml", "w") as f:
    f.write(DOCKER_COMPOSE)

print("Dockerfile and docker-compose.yml written.")
print("\nKey patterns:")
print("  1. Multi-stage build — small image, no build tools in production")
print("  2. Non-root user — reduced attack surface")
print("  3. HEALTHCHECK — Docker restarts unhealthy containers automatically")
print("  4. WEB_CONCURRENCY env var — tune workers per environment without rebuilding")
print("     Formula: 2 * CPU_CORES + 1  (e.g. 4-core machine → 9 workers)")
print("  5. SESSION_TTL_SECONDS env var — configurable session expiry")
print("  6. REDIS_URL wired into agent service — sessions can use Redis")

---
## Part 6: Response Caching for Cost Reduction

Identical or near-identical questions don't need a fresh LLM call. Cache the responses.

In [ ]:
import hashlib
import json
from typing import Optional

class ResponseCache:
    """Simple in-memory cache. Swap for Redis in production."""
    
    def __init__(self, ttl_seconds: int = 3600):
        self._cache: dict[str, tuple[str, float]] = {}  # key → (value, expires_at)
        self.ttl = ttl_seconds
        self.hits = 0
        self.misses = 0
    
    def _key(self, model: str, messages: list) -> str:
        # Cache key is based on the LAST user message only (not full history).
        # This ensures the same question gets a cache hit regardless of what
        # was said earlier in the conversation.
        last_user_msg = next(
            (m["content"] for m in reversed(messages) if m["role"] == "user"), ""
        )
        content = json.dumps({"model": model, "last_user_msg": last_user_msg}, sort_keys=True)
        return hashlib.sha256(content.encode()).hexdigest()[:16]
    
    def get(self, model: str, messages: list) -> Optional[str]:
        key = self._key(model, messages)
        entry = self._cache.get(key)
        if entry and time.time() < entry[1]:
            self.hits += 1
            return entry[0]
        self.misses += 1
        return None
    
    def set(self, model: str, messages: list, response: str):
        key = self._key(model, messages)
        self._cache[key] = (response, time.time() + self.ttl)
    
    @property
    def hit_rate(self) -> float:
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0.0

cache = ResponseCache(ttl_seconds=3600)

async def cached_llm_call(messages: list, model="gpt-4o-mini") -> tuple[str, bool]:
    """Returns (response, was_cached)."""
    cached = cache.get(model, messages)
    if cached:
        return cached, True
    
    response = await aclient.chat.completions.create(model=model, messages=messages)
    result = response.choices[0].message.content
    cache.set(model, messages, result)
    return result, False

# First call — cache miss
messages = [{"role": "user", "content": "What is LangGraph in one sentence?"}]
r1, cached1 = await cached_llm_call(messages)
print(f"Call 1 — cached: {cached1}")
print(f"Response: {r1[:80]}...")

# Second call with same question but different history — still a cache HIT
# because we key on last user message, not full history
messages_with_history = [
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "Hi there!"},
    {"role": "user", "content": "What is LangGraph in one sentence?"},
]
r2, cached2 = await cached_llm_call(messages_with_history)
print(f"\nCall 2 — cached: {cached2} (same question, different history)")
print(f"Cache hit rate: {cache.hit_rate:.1%}")

---
## Production Deployment Checklist

Before shipping an agent to production:

| Category | Check |
|----------|-------|
| **API** | FastAPI with health endpoint at `/health` |
| **Config** | All secrets in env vars, never hardcoded |
| **Concurrency** | Async handlers, N uvicorn workers |
| **Reliability** | Retry logic + model fallback chain |
| **Cost control** | Per-session token budgets + rate limiting |
| **Caching** | Response cache for repeated queries |
| **Streaming** | SSE endpoint for real-time UX |
| **Sessions** | Session history trimmed to last N messages |
| **Container** | Dockerfile with non-root user + healthcheck |
| **Observability** | Logs with session_id + latency (Lab 4 covers this) |

## Where to deploy
- **Fly.io** — easiest for FastAPI, free tier, Docker native
- **Railway** — one-click Docker deploy, auto-scaling
- **AWS Lambda** with Mangum — serverless FastAPI, pay-per-request
- **Azure Container Apps** — great fit if you know .NET/Azure
- **HuggingFace Spaces** — already know this from the course, good for Gradio